# 6 · Analyzing and rewriting rules

**The problem.** The QA team has hundreds of stored rules. Before a schema change, they need to know which rules read
the property being renamed. For reviews, they want every rule written the same way, with the property first. Both are
programs over rules: one reads them, the other rewrites them.

Because a rule is data, those programs are ordinary code over a small tree. This case study writes both, and shows the
tools that make them short: `walk`, `fold`, `same`, and rewriting rules.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Symbolics, Terms } from "@mbse/expressions/Framework";
import { Hole, Pairwise, Pattern, Rule } from "@mbse/expressions/Framework/Translators";
import { show } from "./toolkit.ts";

const [self, c] = [E.variable("this"), E.variable("c")];
const celsius = self.celsius;
const rules: Record<string, any> = {
  in_range: E.literal(2.0).le(celsius).and_(celsius.le(8.0)).data,
  on_time: self.hours.add(E.variable("to_go")).le(72n).data,
  allowed: self.readings.count_where("c", c.gt(8.0)).le(2n).and_(self.readings.max().le(12.0)).data,
  reported: self.has("celsius").data,
};

## Step 1: a rule is a tree of terms

Every term has a **form**: its `kind` (literal, variable, operation, let, quantifier), its `attributes` (a literal's
value, an operation's name) and its ordered `arguments`, which are terms. `Terms.walk` visits each term once:

In [2]:
for (const term of Terms.walk(rules.in_range)) {
  const form = term.form();
  console.log(form.kind.padEnd(10), Object.fromEntries(form.attributes), form.arguments.length);
}

operation  { name: "and" } 2


operation  { name: "le" } 2


literal    { value: 2 } 0


operation  { name: "get" } 2


variable   { name: "this" } 0


literal    { value: "celsius" } 0


operation  { name: "le" } 2


literal    { value: 8 } 0


`get(this, 'celsius')` appears once in the walk, although the rule uses it twice: it's one term, shared, as it was
saved once in case study 1.

## Step 2: which rules read a property

A `get` or `has` whose second argument is a literal names the property it reads. Collecting those is a loop over the
walk:

In [3]:
function reads(rule: any): string[] {
  const found = new Set<string>();
  for (const term of Terms.walk(rule)) {
    const form = term.form();
    if (form.kind === "operation" && ["get", "has"].includes(form.attributes.get("name"))) {
      const name = (form.arguments[1] as any).form();
      if (name.kind === "literal") found.add(name.attributes.get("value"));
    }
  }
  return [...found].sort();
}

console.log(Object.fromEntries(Object.entries(rules).map(([name, rule]) => [name, reads(rule)])));

{
  in_range: [ "celsius" ],
  on_time: [ "hours" ],
  allowed: [ "readings" ],
  reported: [ "celsius" ]
}


Renaming `celsius` touches two of the four rules. `Symbolics.free` answers the other usual question, which
variables a rule needs, and knows that a quantifier's `c` is bound inside it:

In [4]:
console.log(Object.fromEntries(Object.entries(rules).map(([name, rule]) => [name, [...Symbolics.free(rule)].sort()])));

{
  in_range: [ "this" ],
  on_time: [ "this", "to_go" ],
  allowed: [ "this" ],
  reported: [ "this" ]
}


## Step 3: fold, bottom up

`Terms.fold(rule, f)` calls `f(term, results)` once per term, children first, where `results` are what `f` gave for
the term's arguments. Anything computed from the parts is a fold. Here's the toolkit's `show`, in short:

In [5]:
function render(term: any, args: string[]): string {
  const form = term.form();
  const attribute = (key: string): any => form.attributes.get(key);
  if (form.kind === "literal") {
    const value = attribute("value");
    if (typeof value === "string") return `'${value}'`;
    return typeof value === "number" && Number.isInteger(value) ? value.toFixed(1) : String(value);
  }
  if (form.kind === "variable") return attribute("name");
  if (form.kind === "quantifier") return `${attribute("quantifier")}(${args[0]}, ${attribute("name")} => ${args[1]})`;
  return `${attribute("name")}(${args.join(", ")})`;
}

console.log(Terms.fold(rules.allowed, render));
console.log(Terms.fold(rules.allowed, render) === show(rules.allowed));
console.log(Terms.fold(rules.allowed, (_term: any, depths: number[]) => 1 + Math.max(0, ...depths))); // how deep it nests

and(le(count(get(this, 'readings'), c => gt(c, 8.0)), 2), le(max(get(this, 'readings')), 12.0))


true


5


The toolkit's version adds lets and domains. Because each term is folded once, a shared term costs nothing extra.
TypeScript writes `2.0` as `2`, so `render` writes a float's point itself, to show what Python shows.

## Step 4: same

`Terms.same(a, b)` compares two rules by structure: the same kinds, attributes and arguments, all the way down. Two
rules written separately are the same when they say the same thing the same way:

In [6]:
const again = E.literal(2.0).le(self.celsius).and_(self.celsius.le(8.0)).data;
const swapped = self.celsius.ge(2.0).and_(self.celsius.le(8.0)).data;
console.log(Terms.same(rules.in_range, again), Terms.same(rules.in_range, swapped));

true false


## Step 5: rewriting

`2.0 <= celsius` and `celsius >= 2.0` mean the same, and reviews would be easier if every rule used the second. A
**rewriting** is a list of rules, each pairing a pattern with its replacement. Patterns have **holes**, which match any
term (`A`) or any attribute value (`K`, `N`), and a rule is applied wherever its pattern matches. The most specific
pattern wins, so the general rules say what stays as it is, and the specific one what changes. A pattern takes its
attributes first, then its arguments:

In [7]:
const [A, B, K, N] = [new Hole("A"), new Hole("B"), new Hole("K"), new Hole("N", String)];
const keep = [new Rule(new Pattern("variable", { name: N }), new Pattern("variable", { name: N }), "forward"),
  new Rule(new Pattern("literal", { value: K }), new Pattern("literal", { value: K }), "forward"),
  new Rule(new Pattern("operation", { name: N }, A), new Pattern("operation", { name: N }, A), "forward"),
  new Rule(new Pattern("operation", { name: N }, A, B), new Pattern("operation", { name: N }, A, B), "forward")];
const flip = [["le", "ge"], ["lt", "gt"], ["ge", "le"], ["gt", "lt"]].map(([op, mirror]) =>
  new Rule(new Pattern("operation", { name: op }, new Pattern("literal", { value: K }), A),
    new Pattern("operation", { name: mirror }, A, new Pattern("literal", { value: K })), "forward"));
const property_first = new Pairwise(E.DIALECT, E.DIALECT, [...keep, ...flip]);
const rewritten = property_first.forward(rules.in_range);
console.log(show(rewritten as any));
console.log(Terms.same(rewritten, swapped));

and(ge(get(this, 'celsius'), 2.0), le(get(this, 'celsius'), 8.0))


true


`forward` makes a new rule and leaves the old one as it was: expressions are never changed in place. The rewritten
rule is now the same as the one written property first.

The rewriting above handles terms with one or two arguments; `allowed` has a quantifier, which it has no rule for, and
it says so rather than guessing:

In [8]:
try {
  property_first.forward(rules.allowed);
} catch (error) {
  console.log((error as Error).message);
}

Basic quantifier 'count' has no Basic counterpart


## Why it works this way

- **A small tree, the same in every dialect.** Five kinds of term, each with a form, so an analysis is a loop or a fold,
  and it works for any dialect's expressions (case study 7), which have forms too.
- **Rewriting is translation.** A rewriting is a translator from a dialect to itself, written with the same rules as
  the translators between languages, which case study 7 uses.
- **Refuse what isn't covered.** A rewriting that met a term it had no rule for and copied it silently could change
  what a rule means without anyone noticing.